# Enveda CASMI 2026 — Exploratory Data Analytics & Multi-Model Retrieval + Reranking Dashboard

This notebook provides a complete **Google Colab Data Analytics, Dataset Downloader, Secrets Manager, and Multi-Model Retrieval + Reranking Benchmark Suite** for the **Enveda CASMI 2026 – Molecule ID From Mass Spectra** competition.

--- 
## Required & Optional Google Colab Secrets Setup
Before running this notebook in Google Colab, click the **Secrets (🔑 key icon)** in the left sidebar of Colab, toggle **"Notebook access"** ON, and add the following secrets:

| Secret Name | Required? | Description / How to Obtain |
| :--- | :---: | :--- |
| `KAGGLE_USERNAME` | **Yes** | Your Kaggle account username (from `kaggle.json` downloaded via *Kaggle -> Settings -> API -> Create New Token*). |
| `KAGGLE_KEY` | **Yes** | Your 32-character Kaggle API token key (or set `KAGGLE_API_TOKEN` containing either the raw key or full JSON string). |
| `HF_TOKEN` | **Recommended** | Your Hugging Face Read Token (`hf_...`) for pulling open-weights mass spectrometry / chemistry checkpoints (`DreaMS`, `MassSpecGym`, `ChemBERTa`). |
| `KAGGLE_MODEL_DATASET_SLUG` | **Recommended** | Your target Kaggle dataset slug (e.g., `sweeden/casmi26-trained-models`) where trained Colab models are published for offline Kaggle inference. |
| `GITHUB_TOKEN` | *Optional* | GitHub Personal Access Token (`ghp_...`) if you want to push updated benchmark metrics directly from Colab to `sweeden-ttu/casmi-2026`. |
| `WANDB_API_KEY` | *Optional* | Weights & Biases API key for logging GPU training curves and hyperparameter sweeps. |

In [ ]:
# Cell 1: Install Pinned CASMI 2026 Dependencies & Clone/Update Repository if on Colab
!pip install -q rdkit==2026.03.3 lightgbm polars pyarrow kaggle huggingface_hub matplotlib seaborn tqdm

import os
import sys
import shutil
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not Path('/content/casmi-2026').exists():
    !git clone https://github.com/sweeden-ttu/casmi-2026.git /content/casmi-2026

for p in [Path('/content/casmi-2026'), Path('/mnt/LMSTUDIO/CASMI'), Path('.')]:
    if (p / 'casmi26').exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

DRIVE_ROOT = Path('/content/drive/MyDrive/CASMI2026') if IN_COLAB else Path('/mnt/LMSTUDIO/CASMI')
FAST_NVME = Path('/content/casmi_fast') if IN_COLAB else (DRIVE_ROOT / 'data')
for d in [DRIVE_ROOT / 'data', DRIVE_ROOT / 'candidates', DRIVE_ROOT / 'kaggle_dataset_export', FAST_NVME]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Running in Colab: {IN_COLAB} | Persistent Google Drive Root: {DRIVE_ROOT}')

In [ ]:
# Cell 2: Load & Verify Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY, HF_TOKEN, etc.)
from casmi26.benchmark import setup_colab_and_kaggle_secrets, ONLINE_MODEL_ZOO
import pandas as pd

secrets_status = setup_colab_and_kaggle_secrets(drive_root=DRIVE_ROOT)
display(secrets_status.summary_dataframe())

In [ ]:
# Cell 3: Download All Competition Datasets & Candidate Shards to Google Drive
# Downloads:
#   1. `enveda-casmi-2026` (train.parquet, test.parquet)
#   2. `rishavyadav/casmi-2026-candidate-database` (COCONUT 2.0 + PubChem exact-mass Parquet shards)
#   3. Optional open-weights checkpoints (MassSpecGym / DreaMS metadata via Hugging Face if HF_TOKEN is set)

train_parquet_drive = DRIVE_ROOT / 'data' / 'train.parquet'
if not train_parquet_drive.exists() and 'KAGGLE_USERNAME' in secrets_status.loaded_keys:
    print('[1/2] Downloading Enveda CASMI 2026 competition dataset to Google Drive...')
    !kaggle competitions download -c enveda-casmi-2026 -p "{DRIVE_ROOT}/data"
    !unzip -o "{DRIVE_ROOT}/data/*.zip" -d "{DRIVE_ROOT}/data" && rm -f "{DRIVE_ROOT}/data"/*.zip
else:
    print(f'[INFO] Competition train.parquet status on Drive: exists={train_parquet_drive.exists()}')

cand_dir_drive = DRIVE_ROOT / 'candidates'
if not list(cand_dir_drive.rglob('*.parquet')) and 'KAGGLE_USERNAME' in secrets_status.loaded_keys:
    print('[2/2] Downloading CASMI 2026 Candidate Database (COCONUT 2.0 + PubChem shards) to Google Drive...')
    !kaggle datasets download -d rishavyadav/casmi-2026-candidate-database -p "{cand_dir_drive}" --unzip
else:
    print(f'[INFO] Candidate Parquet shards found: {len(list(cand_dir_drive.rglob("*.parquet")))}')

# Stage active Parquet files onto Colab local NVMe (/content/casmi_fast) for 10x faster read speeds
if IN_COLAB and train_parquet_drive.exists() and not (FAST_NVME / 'train.parquet').exists():
    shutil.copy2(train_parquet_drive, FAST_NVME / 'train.parquet')
    if (DRIVE_ROOT / 'data' / 'test.parquet').exists():
        shutil.copy2(DRIVE_ROOT / 'data' / 'test.parquet', FAST_NVME / 'test.parquet')

In [ ]:
# Cell 4: Load Dataset (or Synthetic Multi-Adduct Benchmark if Offline) & Run Exploratory Analytics
import json
import numpy as np
from casmi26.benchmark import compute_exploratory_analytics, run_retrieval_reranking_benchmark
from casmi26.retrieval import build_candidate_pool_from_sources
from casmi26.validation import assign_leak_free_folds, sample_timstof_like_validation_panel

active_train_path = (FAST_NVME / 'train.parquet') if (FAST_NVME / 'train.parquet').exists() else train_parquet_drive
if active_train_path.exists():
    print(f'[OK] Loading real competition dataset from {active_train_path}...')
    train_df = pd.read_parquet(active_train_path)
    cand_shards = sorted(cand_dir_drive.rglob('*.parquet'))
else:
    print('[INFO] Full train.parquet not yet downloaded; generating realistic multi-spectrum benchmark dataset...')
    from tests.test_pipeline import generate_synthetic_casmi26_data
    train_df, cand_df = generate_synthetic_casmi26_data(seed=2026)
    synth_dir = Path('/mnt/LMSTUDIO/CASMI/data/synthetic') if Path('/mnt/LMSTUDIO/CASMI').exists() else Path('data/synthetic')
    synth_dir.mkdir(parents=True, exist_ok=True)
    cand_df[cand_df['is_coconut']].to_parquet(synth_dir / 'coconut_candidates.parquet', index=False)
    cand_df[~cand_df['is_coconut']].to_parquet(synth_dir / 'pubchem_candidates.parquet', index=False)
    cand_shards = [synth_dir / 'coconut_candidates.parquet', synth_dir / 'pubchem_candidates.parquet']

train_df['fold'] = assign_leak_free_folds(train_df, n_splits=5, group_col='inchikey14')
tr_split = train_df[train_df['fold'] != 0].reset_index(drop=True)
va_split = train_df[train_df['fold'] == 0].reset_index(drop=True)

candidate_pool = build_candidate_pool_from_sources(train_df=tr_split, candidate_parquet_paths=cand_shards)
eda_summary = compute_exploratory_analytics(train_df, candidate_pool=candidate_pool)
print(json.dumps(eda_summary, indent=2))

In [ ]:
# Cell 5: Online Retrieval & Reranking Model Zoo Registry
zoo_df = pd.DataFrame(ONLINE_MODEL_ZOO)
display(zoo_df)

In [ ]:
# Cell 6: Train & Benchmark All 6 Retrieval + Reranking Models Across Class 2 (Isomers) & Class 1+2
from casmi26.features import clean_and_normalize_spectrum, smiles_to_fingerprint, spectrum_to_binned_vector
from casmi26.models import CASMIPipeline, SpectralLibraryIndex, SpectrumToFingerprintModel

max_train = min(len(tr_split), 150_000)
tr_sub = tr_split.sample(n=max_train, random_state=42) if len(tr_split) > max_train else tr_split

X_binned = np.zeros((len(tr_sub), 1536), dtype=np.float32)
X_nm = np.zeros(len(tr_sub), dtype=np.float32)
X_pos = np.zeros(len(tr_sub), dtype=np.float32)
Y_fp = np.zeros((len(tr_sub), 2215), dtype=np.float32)
Y_hard_neg_fp = np.zeros((len(tr_sub), 2215), dtype=np.float32)

for i, row in enumerate(tr_sub.itertuples(index=False)):
    c_mzs, c_probs = clean_and_normalize_spectrum(row.ms2_mzs, row.ms2_normalized_intensities, row.precursor_mz)
    X_binned[i] = spectrum_to_binned_vector(c_mzs, c_probs, row.precursor_mz)
    X_nm[i] = float(row.precursor_mz)
    X_pos[i] = 1.0 if str(row.ionization_mode).startswith('pos') else 0.0
    Y_fp[i] = smiles_to_fingerprint(row.normalized_smiles)
    hard_isomers = candidate_pool.sample_hard_isomer_negatives(float(row.precursor_mz), str(row.inchikey14), k_negatives=4)
    if hard_isomers:
        Y_hard_neg_fp[i] = np.mean([smiles_to_fingerprint(s) for s in hard_isomers], axis=0)

fp_model = SpectrumToFingerprintModel(use_torch_if_available=True, ridge_alpha=5.0)
fp_model.fit(X_binned, X_nm, X_pos, Y_fp, hard_neg_fps=Y_hard_neg_fp, epochs=15, batch_size=256, lr=1e-3)

# 1. Class 2 Benchmark (Out-of-library isomers: query molecules withheld from reference spectral library)
val_queries, val_truth = sample_timstof_like_validation_panel(va_split, n_molecules=100)
spec_index_class2 = SpectralLibraryIndex.build(tr_split)
pipeline = CASMIPipeline(
    candidate_pool=candidate_pool,
    spectral_index=spec_index_class2,
    fp_model=fp_model,
    ppm_tolerance=15.0,
)
pipeline.fit_reranker(val_queries, val_truth)
df_bench_class2 = run_retrieval_reranking_benchmark(
    pipeline, val_queries, val_truth, class_label='Class 2 (Out-of-Library Isomers)'
)

# 2. Class 1+2 Combined Benchmark (Full reference library + candidate pool)
pipeline.spectral_index = SpectralLibraryIndex.build(train_df)
df_bench_combined = run_retrieval_reranking_benchmark(
    pipeline, val_queries, val_truth, class_label='Class 1+2 (Combined Library + Isomers)'
)

full_benchmark_df = pd.concat([df_bench_class2, df_bench_combined], ignore_index=True)
display(full_benchmark_df)